<a href="https://colab.research.google.com/github/jahidurmahim/Machine_Learning_Laboratory/blob/main/Underoverfit_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/job_salary_prediction_dataset.csv')

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.isna().transpose(), cmap="YlGnBu", cbar_kws={'label': 'Missing Data'})
plt.title('Missing Data in the Dataset')
plt.show()

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns
print(f"Categorical columns to encode: {list(categorical_cols)}")

In [ ]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
display(df_encoded.head())

In [ ]:
print(f"Shape of the DataFrame after one-hot encoding: {df_encoded.shape}")
print(f"Columns of the DataFrame after one-hot encoding: {df_encoded.columns.tolist()}")

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target variable (y)
X = df_encoded.drop('salary', axis=1)
y = df_encoded['salary']

# Split data into training (70%) and temporary (30%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Split temporary data into validation (15%) and test (15%) sets
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of y_val: {y_val.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Select limited features for underfitting
selected_features = [
    'experience_years',
    'skills_count',
    'certifications',
    'job_title_Software Engineer',
    'remote_work_Yes'
]

X_train_limited = X_train[selected_features].astype(int)
X_val_limited = X_val[selected_features].astype(int)
X_test_limited = X_test[selected_features].astype(int)

print(f"Shape of X_train with limited features: {X_train_limited.shape}")
print(f"Shape of X_val with limited features: {X_val_limited.shape}")
print(f"Shape of X_test with limited features: {X_test_limited.shape}")

In [ ]:
# Define a small neural network model with 1 hidden layer
model_underfit = keras.Sequential([
    layers.Dense(units=10, activation='relu', input_shape=(X_train_limited.shape[1],)), # One hidden layer
    layers.Dense(units=1) # Output layer for regression (predicting salary)
])

# Compile the model
model_underfit.compile(optimizer='adam',
                      loss='mean_squared_error',
                      metrics=['mean_absolute_error'])

# Train the model for a very few epochs (e.g., 5 epochs)
history_underfit = model_underfit.fit(X_train_limited, y_train,
                                    epochs=5, # Very few epochs to ensure underfitting
                                    validation_data=(X_val_limited, y_val),
                                    verbose=1) # Show training progress

# Evaluate the model on the test set
loss, mae = model_underfit.evaluate(X_test_limited, y_test, verbose=0)
print(f"\nUnderfitting Model - Test Loss (MSE): {loss:.2f}")
print(f"Underfitting Model - Test MAE: {mae:.2f}")

The model has been trained. A high loss and MAE on both training and validation sets, and a small gap between them, are indicators of underfitting. Let's visualize the training history.

In [ ]:
# Plot training & validation loss values to visualize underfitting
plt.figure(figsize=(10, 6))
plt.plot(history_underfit.history['loss'], label='Training Loss')
plt.plot(history_underfit.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training (Underfitting)')
plt.xlabel('Epoch')
plt.ylabel('Loss (Mean Squared Error)')
plt.legend()
plt.grid(True)
plt.show()

### PHASE 2: CREATE OVERFITTING

To achieve overfitting, we will:

*   Use **all available features** from the dataset.
*   Define a **more complex neural network** with multiple hidden layers and more neurons.
*   Train the model for a **larger number of epochs** (e.g., 50).
*   Ensure **no regularization or dropout** is applied to encourage the model to memorize the training data.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define a more complex neural network model with multiple hidden layers and more neurons
model_overfit = keras.Sequential([
    layers.Dense(units=256, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(units=128, activation='relu'),
    layers.Dense(units=64, activation='relu'),
    layers.Dense(units=1) # Output layer for regression
])

# Compile the model with an appropriate optimizer and loss function
model_overfit.compile(optimizer='adam',
                     loss='mean_squared_error',
                     metrics=['mean_absolute_error'])

model_overfit.summary()

In [ ]:
# Train the model for many epochs (e.g., 50 epochs)
history_overfit = model_overfit.fit(X_train.astype('float32'), y_train,
                                  epochs=50, # Many epochs to encourage overfitting
                                  validation_data=(X_val.astype('float32'), y_val),
                                  verbose=1) # Show training progress

# Evaluate the model on the test set
loss_overfit, mae_overfit = model_overfit.evaluate(X_test.astype('float32'), y_test, verbose=0)
print(f"\nOverfitting Model - Test Loss (MSE): {loss_overfit:.2f}")
print(f"Overfitting Model - Test MAE: {mae_overfit:.2f}")

The model has been trained for an extended period with a complex architecture and all features. We expect to see a very low training loss but a high validation loss, indicating overfitting. Let's visualize the training history.

### Forcing Overfitting: Reducing Training Data Size

To explicitly force overfitting, we will significantly reduce the size of the training dataset. We will take only 5000 samples from the `X_train` and `y_train` datasets, while keeping the validation and test sets the same.

In [ ]:
# Re-define the complex neural network model for explicit overfitting (using reduced training data and more epochs)
# (The model architecture is already defined in the notebook as 'model_overfit')

# Train the model with the reduced dataset for more epochs (e.g., 100 epochs)
history_forced_overfit = model_overfit.fit(X_train_overfit.astype('float32'), y_train_overfit,
                                          epochs=100, # Increased epochs
                                          validation_data=(X_val.astype('float32'), y_val),
                                          verbose=1) # Show training progress

# Evaluate the model on the test set
loss_forced_overfit, mae_forced_overfit = model_overfit.evaluate(X_test.astype('float32'), y_test, verbose=0)
print(f"\nForced Overfitting Model - Test Loss (MSE): {loss_forced_overfit:.2f}")
print(f"Forced Overfitting Model - Test MAE: {mae_forced_overfit:.2f}")

The model has now been trained with a significantly smaller dataset and for a large number of epochs. We expect to see a more pronounced overfitting effect compared to the previous 'overfitting' phase. Let's visualize the training history.

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots side-by-side
fig, ax = plt.subplots(1, 2, figsize=(20, 7)) # 1 row, 2 columns

# Plot Training & Validation Loss values on the first subplot
ax[0].plot(history_forced_overfit.history['loss'], label='Training Loss')
ax[0].plot(history_forced_overfit.history['val_loss'], label='Validation Loss')
ax[0].set_title('Model Loss During Training (Forced Overfitting)')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss (Mean Squared Error)')
ax[0].legend()
ax[0].grid(True)

# Plot Training & Validation MAE values on the second subplot
ax[1].plot(history_forced_overfit.history['mean_absolute_error'], label='Training MAE')
ax[1].plot(history_forced_overfit.history['val_mean_absolute_error'], label='Validation MAE')
ax[1].set_title('Model MAE During Training (Forced Overfitting)')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Mean Absolute Error')
ax[1].legend()
ax[1].grid(True)

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

In [ ]:
# Reduce training dataset size to 5000 samples
X_train_overfit = X_train.sample(n=5000, random_state=42)
y_train_overfit = y_train.sample(n=5000, random_state=42)

print(f"Shape of X_train for overfitting: {X_train_overfit.shape}")
print(f"Shape of y_train for overfitting: {y_train_overfit.shape}")
print(f"Shape of X_val (unchanged): {X_val.shape}")
print(f"Shape of y_val (unchanged): {y_val.shape}")